In [1]:
"""
steps for LDA for different languages:
1. import everything and define BCTs
2. figure out how to break up segments for LDA to run on in each language (Stopwords ISO)
3.
"""


'\nsteps for LDA for different languages:\n1. import everything and define BCTs\n2. figure out how to break up segments for LDA to run on in each language (Stopwords ISO)\n3.\n'

In [2]:
# Imports

!pip install datasets scikit-learn nltk stopwordsiso
import re
import time
import json
from datasets import load_dataset
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')
import stopwordsiso



 # log into HF
#from google.colab import userdata
#import os
#os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN') ### if you have one please use your hugging face token if not please comment out

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.9/74.9 kB 4.9 MB/s eta 0:00:00


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [3]:
# Save BCTs as dictionary -> final BCTs looked at by native speakers but it's not actually that simple
BCT_FORMS = {
    'en': {
        'black':  ['black'],
        'white':  ['white'],
        'red':    ['red'],
        'green':  ['green'],
        'yellow': ['yellow'],
        'blue':   ['blue'],
        'brown':  ['brown'],
        'orange': ['orange'],
        'pink':   ['pink'],
        'purple': ['purple'],
        'gray':   ['gray', 'grey'],
    },
    'fr': {
        'black':  ['noir', 'noire', 'noirs', 'noires'],
        'white':  ['blanc', 'blanche', 'blancs', 'blanches'],
        'red':    ['rouge', 'rouges'],
        'green':  ['vert', 'verte', 'verts', 'vertes'],
        'yellow': ['jaune', 'jaunes'],
        'blue':   ['bleu', 'bleue', 'bleus', 'bleues'],
        'brown':  ['marron', 'marrons'],
        'orange': ['orange', 'oranges'],
        'pink':   ['rose', 'roses'],
        'purple': ['violet', 'violette', 'violets', 'violettes'],
        'gray':   ['gris', 'grise', 'grises'],
    },
    'es': {
        'black':  ['negro', 'negra', 'negros', 'negras'],
        'white':  ['blanco', 'blanca', 'blancos', 'blancas'],
        'red':    ['rojo', 'roja', 'rojos', 'rojas'],
        'green':  ['verde', 'verdes'],
        'yellow': ['amarillo', 'amarilla', 'amarillos', 'amarillas'],
        'blue':   ['azul', 'azules'],
        'brown':  ['marrón', 'marrones'],
        'orange': ['naranja', 'naranjas'],
        'pink':   ['rosa', 'rosas'],
        'purple': ['morado', 'morada', 'morados', 'moradas'],
        'gray':   ['gris', 'grises'],
    },
    'ca': {
        'black':  ['negre', 'negra', 'negres'],
        'white':  ['blanc', 'blanca', 'blancs', 'blanques'],
        'red':    ['vermell', 'vermella', 'vermells', 'vermelles'],
        'green':  ['verd', 'verda', 'verds', 'verdes'],
        'yellow': ['groc', 'groga', 'grocs', 'grogues'],
        'blue':   ['blau', 'blava', 'blaus', 'blaves'],
        'brown':  ['marró', 'marrons'],
        'orange': ['taronja', 'taronges'],
        'pink':   ['rosa', 'roses'],
        'purple': ['morat', 'morada', 'morats', 'morades'],
        'gray':   ['gris', 'grisa', 'grisos', 'grises'],
    },
}

# link language to wikipedia dumps:
WIKI_CONFIG = {
    'en': '20231101.en',
    'fr': '20231101.fr',
    'es': '20231101.es',
    'ca': '20231101.ca',
}


# get list of stopwords -> (non-meaning making words e.g. of, a, on)

STOPWORDSISO_CODE = {'en': 'en', 'fr': 'fr', 'es': 'es', 'ca': 'ca'}

def get_stopwords(lang):
    return list(stopwordsiso.stopwords(STOPWORDSISO_CODE[lang]))




In [4]:
""" set parameters:
max number of instances for the colour,
minimum sentence length -> 20 characters not words,
absolutely maximum number of articles for it to look through,
shuffle buffer to keep order of things random
random seed so it can be reproduced
show progress every 500 so I know it's not stuck
number of topics for the lda to show. chose 15 as I knew I would be doing it by hand,
 so long enough to hopefully see patterns but not long enough that its impossible to do by hand.
"""
RUN_MODE = 'full'
if RUN_MODE == 'full':
    N_SENTENCES_TARGET = 1500
    MIN_SENTENCE_LEN = 20
    ARTICLE_SCAN_CAP = 400_000
    SHUFFLE_BUFFER = 2000
    RANDOM_SEED = 42
    PROGRESS_EVERY = 500
    N_TOPICS = 15
else:
    raise ValueError(f"Unknown RUN_MODE: {RUN_MODE!r} (expected'full')")

print(f"RUN_MODE={RUN_MODE!r}  |  N_SENTENCES_TARGET={N_SENTENCES_TARGET}  "
      f"ARTICLE_SCAN_CAP={ARTICLE_SCAN_CAP}  N_TOPICS={N_TOPICS}")



RUN_MODE='full'  |  N_SENTENCES_TARGET=1500  ARTICLE_SCAN_CAP=400000  N_TOPICS=15


In [5]:
"""
This is for counting the proper nouns:
(1) build alternative forms of colors e.g, rojo, roja, rojos into one color_form
(2) take any of the alt forms and see if there are any capitals after it
(3) take any of the alt forms and see if there is a capital for the word before.
(4) return them if this function is called.
"""
def build_cap_patterns(color_forms):
    # Build expression catching 'Color Capitalized' and 'Capitalized Color'.
    alt = "|".join(re.escape(f) for f in color_forms)
    color_then_cap = re.compile(rf"\b(?i:{alt}) [A-ZÀ-Ý][\wÀ-ÿ]+", re.UNICODE)
    cap_then_color = re.compile(rf"\b[A-ZÀ-Ý][\wÀ-ÿ]+ (?i:{alt})\b", re.UNICODE)
    return color_then_cap, cap_then_color

"""
this cell only works for cap_then_color because for color then cap we know this is not the beginning of a sentence
(1) get rid of spaces at the beginning or end of a sentence
(2) for the cap_then_color check whether the capital occurs at NOT the first 'position' or is NOT the first character in the sentence
(3) If that is true then it is a proper noun.
(4) search the colour then cap as well.
"""
def is_proper_noun_flagged(raw_sentence, cap_then_color_re, color_then_cap_re):
    stripped = raw_sentence.strip()
    for m in cap_then_color_re.finditer(stripped):
        if m.start() != 0:
            return True
    return bool(color_then_cap_re.search(stripped))

#build a dictionary to hold (per language) the functions to search all colours for proper nouns
CAP_PATTERNS = {
    lang: {
        color: build_cap_patterns(forms)
        for color, forms in colors.items()
    }
    for lang, colors in BCT_FORMS.items()
}





In [6]:
"""Collect all the sentences that qualify!
(1) define funcrtion with parameters decided earleir
(2) call ditionary for cap patterns
(3) show where we are looking (Wikipedia) per language
(4) shuffle articles so we don't search alphabetically
(5) split by newline, ., !, ? and check the character count of the sentence.
(6) print out progress as it searches
(7) run on each language
"""

def collect_all_colors(lang, n_target=N_SENTENCES_TARGET,
                        min_len=MIN_SENTENCE_LEN,
                        scan_cap=ARTICLE_SCAN_CAP,
                        shuffle_buffer=SHUFFLE_BUFFER,
                        seed=RANDOM_SEED,
                        progress_every=PROGRESS_EVERY):

    color_forms = BCT_FORMS[lang]
    cap_patterns = CAP_PATTERNS[lang]


    results = {color: [] for color in color_forms}
    counts_rejected_short = 0
    articles_scanned = 0

    config_name = WIKI_CONFIG[lang]
    print(f"[{lang}] Loading stream: wikimedia/wikipedia/{config_name}")
    dataset = load_dataset("wikimedia/wikipedia", config_name, split="train", streaming=True)
    dataset = dataset.shuffle(seed=seed, buffer_size=shuffle_buffer)

    def all_full():
        return all(len(v) >= n_target for v in results.values())

    start = time.time()
    for article in dataset:
        articles_scanned += 1
        text = article['text']
        raw_sentences = re.split(r'[.!?\n]+', text)

        for raw in raw_sentences:
            if len(raw) <= min_len:
                counts_rejected_short += 1
                continue
            lower = raw.lower()

            for color, forms in color_forms.items():
                if len(results[color]) >= n_target:
                    continue

                hit = any(re.search(rf"\b{re.escape(f)}\b", lower) for f in forms)
                if not hit:
                    continue

                color_then_cap_re, cap_then_color_re = cap_patterns[color]
                flagged = is_proper_noun_flagged(raw, cap_then_color_re, color_then_cap_re)

                results[color].append({"text": lower, "is_proper_noun": flagged})

        if articles_scanned % progress_every == 0:
            fill = {c: len(v) for c, v in results.items()}
            elapsed = time.time() - start
            print(f"[{lang}] scanned={articles_scanned} elapsed={elapsed:.0f}s fill={fill}")

        if all_full() or articles_scanned >= scan_cap:
            break

    print(f"[{lang}] DONE. articles_scanned={articles_scanned}, "
          f"rejected_short={counts_rejected_short}")
    for color, sents in results.items():
        n_flagged = sum(s["is_proper_noun"] for s in sents)
        print(f"    {color:8s}: n={len(sents):5d}  proper_noun_flagged={n_flagged}")

    return results, {"articles_scanned": articles_scanned, "lang": lang}



In [7]:
results_en, meta_en = collect_all_colors('en')

[en] Loading stream: wikimedia/wikipedia/20231101.en


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


README.md:   0%|          | 0.00/131k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/41 [00:00<?, ?it/s]

[en] scanned=500 elapsed=2s fill={'black': 168, 'white': 57, 'red': 40, 'green': 20, 'yellow': 8, 'blue': 25, 'brown': 20, 'orange': 11, 'pink': 5, 'purple': 3, 'gray': 9}
[en] scanned=1000 elapsed=3s fill={'black': 223, 'white': 86, 'red': 57, 'green': 31, 'yellow': 13, 'blue': 38, 'brown': 25, 'orange': 17, 'pink': 5, 'purple': 4, 'gray': 14}
[en] scanned=1500 elapsed=4s fill={'black': 267, 'white': 133, 'red': 91, 'green': 68, 'yellow': 23, 'blue': 52, 'brown': 54, 'orange': 17, 'pink': 9, 'purple': 8, 'gray': 17}
[en] scanned=2000 elapsed=5s fill={'black': 294, 'white': 164, 'red': 127, 'green': 101, 'yellow': 30, 'blue': 73, 'brown': 82, 'orange': 20, 'pink': 10, 'purple': 8, 'gray': 22}
[en] scanned=2500 elapsed=6s fill={'black': 337, 'white': 204, 'red': 148, 'green': 130, 'yellow': 36, 'blue': 82, 'brown': 103, 'orange': 34, 'pink': 12, 'purple': 10, 'gray': 34}
[en] scanned=3000 elapsed=7s fill={'black': 377, 'white': 239, 'red': 193, 'green': 152, 'yellow': 37, 'blue': 109, '

In [8]:
results_fr, meta_fr = collect_all_colors('fr')

[fr] Loading stream: wikimedia/wikipedia/20231101.fr


Resolving data files:   0%|          | 0/17 [00:00<?, ?it/s]

[fr] scanned=500 elapsed=9s fill={'black': 51, 'white': 64, 'red': 44, 'green': 18, 'yellow': 6, 'blue': 11, 'brown': 0, 'orange': 4, 'pink': 9, 'purple': 6, 'gray': 4}
[fr] scanned=1000 elapsed=15s fill={'black': 125, 'white': 135, 'red': 68, 'green': 29, 'yellow': 12, 'blue': 22, 'brown': 0, 'orange': 7, 'pink': 21, 'purple': 7, 'gray': 10}
[fr] scanned=1500 elapsed=21s fill={'black': 170, 'white': 202, 'red': 97, 'green': 40, 'yellow': 26, 'blue': 35, 'brown': 2, 'orange': 9, 'pink': 26, 'purple': 9, 'gray': 10}
[fr] scanned=2000 elapsed=25s fill={'black': 227, 'white': 262, 'red': 140, 'green': 55, 'yellow': 37, 'blue': 37, 'brown': 4, 'orange': 16, 'pink': 37, 'purple': 9, 'gray': 23}
[fr] scanned=2500 elapsed=28s fill={'black': 303, 'white': 324, 'red': 171, 'green': 61, 'yellow': 38, 'blue': 41, 'brown': 5, 'orange': 17, 'pink': 46, 'purple': 9, 'gray': 26}
[fr] scanned=3000 elapsed=30s fill={'black': 349, 'white': 385, 'red': 209, 'green': 66, 'yellow': 42, 'blue': 49, 'brown':

In [9]:
results_es, meta_es = collect_all_colors('es')

[es] Loading stream: wikimedia/wikipedia/20231101.es
[es] scanned=500 elapsed=15s fill={'black': 187, 'white': 217, 'red': 276, 'green': 168, 'yellow': 47, 'blue': 136, 'brown': 19, 'orange': 23, 'pink': 88, 'purple': 16, 'gray': 34}
[es] scanned=1000 elapsed=29s fill={'black': 420, 'white': 520, 'red': 529, 'green': 296, 'yellow': 108, 'blue': 267, 'brown': 28, 'orange': 39, 'pink': 164, 'purple': 34, 'gray': 68}
[es] scanned=1500 elapsed=43s fill={'black': 725, 'white': 794, 'red': 818, 'green': 459, 'yellow': 168, 'blue': 413, 'brown': 43, 'orange': 61, 'pink': 257, 'purple': 49, 'gray': 102}
[es] scanned=2000 elapsed=57s fill={'black': 1046, 'white': 1159, 'red': 1011, 'green': 547, 'yellow': 216, 'blue': 482, 'brown': 52, 'orange': 78, 'pink': 405, 'purple': 56, 'gray': 126}
[es] scanned=2500 elapsed=70s fill={'black': 1367, 'white': 1500, 'red': 1257, 'green': 690, 'yellow': 284, 'blue': 564, 'brown': 70, 'orange': 119, 'pink': 505, 'purple': 70, 'gray': 156}
[es] scanned=3000 el

In [10]:
results_ca, meta_ca = collect_all_colors('ca')

[ca] Loading stream: wikimedia/wikipedia/20231101.ca
[ca] scanned=500 elapsed=5s fill={'black': 23, 'white': 27, 'red': 12, 'green': 21, 'yellow': 4, 'blue': 7, 'brown': 4, 'orange': 1, 'pink': 3, 'purple': 0, 'gray': 3}
[ca] scanned=1000 elapsed=6s fill={'black': 46, 'white': 37, 'red': 18, 'green': 30, 'yellow': 13, 'blue': 11, 'brown': 6, 'orange': 3, 'pink': 10, 'purple': 0, 'gray': 4}
[ca] scanned=1500 elapsed=8s fill={'black': 80, 'white': 80, 'red': 29, 'green': 55, 'yellow': 19, 'blue': 16, 'brown': 8, 'orange': 5, 'pink': 19, 'purple': 1, 'gray': 4}
[ca] scanned=2000 elapsed=11s fill={'black': 162, 'white': 115, 'red': 35, 'green': 66, 'yellow': 22, 'blue': 20, 'brown': 8, 'orange': 7, 'pink': 27, 'purple': 3, 'gray': 6}
[ca] scanned=2500 elapsed=14s fill={'black': 202, 'white': 135, 'red': 40, 'green': 78, 'yellow': 23, 'blue': 35, 'brown': 10, 'orange': 7, 'pink': 31, 'purple': 3, 'gray': 8}
[ca] scanned=3000 elapsed=16s fill={'black': 238, 'white': 166, 'red': 98, 'green': 

In [11]:
# collecte results in a .json so we don't have to rerun everything if the colab disconnects.
ALL_RESULTS = {'en': results_en, 'fr': results_fr, 'es': results_es, 'ca': results_ca}
ALL_META = {'en': meta_en, 'fr': meta_fr, 'es': meta_es, 'ca': meta_ca}

with open('bct_sentences_checkpoint.json', 'w', encoding='utf-8') as f:
    json.dump(ALL_RESULTS, f, ensure_ascii=False)

with open('bct_meta_checkpoint.json', 'w', encoding='utf-8') as f:
    json.dump(ALL_META, f, ensure_ascii=False, indent=2)

print("Checkpoint saved: bct_sentences_checkpoint.json, bct_meta_checkpoint.json")



Checkpoint saved: bct_sentences_checkpoint.json, bct_meta_checkpoint.json


In [12]:
# Fit the LDA
def fit_lda_for(sentences_with_flags, lang, n_topics=N_TOPICS, max_features=500, seed=RANDOM_SEED):
    """
    Fit one LDA model on ALL sentences for a color/language (proper nouns
    included). is_proper_noun is included but just used to count how many proper nouns there are
    """
    texts = [s["text"] for s in sentences_with_flags]
    n_proper = sum(1 for s in sentences_with_flags if s["is_proper_noun"])

    if len(texts) < n_topics * 2:
        return None  # not enough data to fit meaningfully

    sw = get_stopwords(lang)
    vectorizer = CountVectorizer(stop_words=sw, max_features=max_features)
    X = vectorizer.fit_transform(texts)

    lda = LatentDirichletAllocation(n_components=n_topics, random_state=seed)
    lda.fit(X)

    words = vectorizer.get_feature_names_out()
    topics = []
    for topic in lda.components_:
        top_words = [words[j] for j in topic.argsort()[-10:]][::-1]
        topics.append(top_words)

    return {"n_docs": len(texts), "n_proper_noun_flagged": n_proper, "topics": topics}

In [13]:
LDA_RESULTS = {}  # GET RESULTS!!

for lang, color_dict in ALL_RESULTS.items():
    for color, sentences in color_dict.items():
        key = (lang, color)
        out = fit_lda_for(sentences, lang=lang)
        LDA_RESULTS[key] = out
        status = "OK" if out else "SKIPPED (too few docs)"
        n_docs = out["n_docs"] if out else 0
        n_proper = out["n_proper_noun_flagged"] if out else 0
        pct = f"{100 * n_proper / n_docs:.0f}%" if out and n_docs else "n/a"
        print(f"{lang}/{color:8s} -> {status} (n={n_docs}, proper_noun_flagged={n_proper} [{pct}])")



/usr/local/lib/python3.12/dist-packages/sklearn/feature_extraction/text.py:402: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['ain', 'daren', 'hadn', 'herse', 'himse', 'itse', 'mayn', 'mightn', 'mon', 'mustn', 'myse', 'needn', 'oughtn', 'shan'] not in stop_words.
  warnings.warn(


en/black    -> OK (n=1500, proper_noun_flagged=687 [46%])
en/white    -> OK (n=1500, proper_noun_flagged=464 [31%])
en/red      -> OK (n=1500, proper_noun_flagged=955 [64%])
en/green    -> OK (n=1500, proper_noun_flagged=741 [49%])
en/yellow   -> OK (n=1500, proper_noun_flagged=391 [26%])
en/blue     -> OK (n=1500, proper_noun_flagged=885 [59%])
en/brown    -> OK (n=1500, proper_noun_flagged=778 [52%])
en/orange   -> OK (n=1500, proper_noun_flagged=730 [49%])
en/pink     -> OK (n=1500, proper_noun_flagged=464 [31%])
en/purple   -> OK (n=1500, proper_noun_flagged=534 [36%])
en/gray     -> OK (n=1500, proper_noun_flagged=553 [37%])


/usr/local/lib/python3.12/dist-packages/sklearn/feature_extraction/text.py:402: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['quelqu'] not in stop_words.
  warnings.warn(


fr/black    -> OK (n=1500, proper_noun_flagged=226 [15%])
fr/white    -> OK (n=1500, proper_noun_flagged=177 [12%])
fr/red      -> OK (n=1500, proper_noun_flagged=326 [22%])
fr/green    -> OK (n=1500, proper_noun_flagged=340 [23%])
fr/yellow   -> OK (n=1500, proper_noun_flagged=499 [33%])
fr/blue     -> OK (n=1500, proper_noun_flagged=319 [21%])
fr/brown    -> OK (n=625, proper_noun_flagged=39 [6%])
fr/orange   -> OK (n=1500, proper_noun_flagged=267 [18%])
fr/pink     -> OK (n=1500, proper_noun_flagged=407 [27%])
fr/purple   -> OK (n=1500, proper_noun_flagged=504 [34%])
fr/gray     -> OK (n=1500, proper_noun_flagged=175 [12%])
es/black    -> OK (n=1500, proper_noun_flagged=238 [16%])
es/white    -> OK (n=1500, proper_noun_flagged=409 [27%])
es/red      -> OK (n=1500, proper_noun_flagged=469 [31%])
es/green    -> OK (n=1500, proper_noun_flagged=367 [24%])
es/yellow   -> OK (n=1500, proper_noun_flagged=74 [5%])
es/blue     -> OK (n=1500, proper_noun_flagged=271 [18%])
es/brown    -> OK (